# Causal Inference Limitations: Case Studies

In prior weeks, we learned methods for making causal claims: DiD, RDD, and IV each exploit a specific source of exogenous variation. This week we ask: **what happens when those methods aren't available?**

Bailard et al. (2024, APSR) study whether Proud Boys online messaging predicts offline violence. They have rich data (514K Telegram messages, 31 months of offline events) but no experiment, no instrument, no discontinuity. They use **Granger causality** (does past X help predict future Y?) instead.

This notebook lets you explore that data and see the difference between prediction and causation:

1. Load and explore the Proud Boys Telegram + offline events data
2. Reproduce the key finding: grievance messages predict offline violence
3. Check the reverse direction: does violence predict grievances?
4. Ask: what would we need (IV? RCT? natural experiment?) to go from prediction to causation?

Data source: [Harvard Dataverse](https://doi.org/10.7910/DVN/OAOJQZ) (CC0 license)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. The Data: Proud Boys Telegram Messages + Offline Events

Bailard et al. (2024) scraped 514,000 messages from Proud Boys Telegram channels (Jan 2020 to July 2022). They trained ML classifiers to categorize each message:

- **Diagnostic**: identifying grievances, naming enemies
- **Prognostic**: proposing solutions, calls to action
- **Motivational**: appeals to group pride, solidarity
- **Othering**: dehumanizing language about outgroups

They then matched weekly message counts to the US Crisis Monitor's data on offline events (protests and violent incidents).

In [ ]:
# Load weekly time-series data (134 weeks)
DATA_URL = 'https://raw.githubusercontent.com/Persuasion-at-Scale/causal-limitations/main/data/bailard-ts-data.tab'
LOCAL_PATH = '../data/bailard-ts-data.tab'

import os
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH, sep='\t')
else:
    df = pd.read_csv(DATA_URL, sep='\t')

df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

print(f"Weeks: {len(df)} (from {df.start_date.min().date()} to {df.end_date.max().date()})")
print(f"Total messages classified: {df.number_texts_all.sum():,.0f}")
print(f"Total violent events: {df.violent_event.sum():.0f}")
print(f"Total non-violent protests: {df.non_violent_protest.sum():.0f}")
print()
print("Message type proportions (weekly averages):")
for col in ['diagnostic', 'prognostic', 'motivational', 'othering']:
    print(f"  {col}: {df[col].mean():.3f}")

## 2. What do the time series look like?

In [ ]:
# Plot message types and offline events over time
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top: message composition over time
for col, color, label in [('diagnostic', 'firebrick', 'Grievances'),
                            ('motivational', 'steelblue', 'Group pride'),
                            ('othering', 'darkorange', 'Othering')]:
    axes[0].plot(df['start_date'], df[col], color=color, label=label, alpha=0.7)
axes[0].set_ylabel('Share of messages')
axes[0].set_title('Proud Boys Telegram: message types over time')
axes[0].legend(fontsize=10)

# Bottom: offline events
axes[1].bar(df['start_date'], df['violent_event'], width=5, color='firebrick',
            alpha=0.7, label='Violent events')
axes[1].bar(df['start_date'], df['non_violent_protest'], width=5, color='steelblue',
            alpha=0.4, label='Non-violent protests', bottom=df['violent_event'])
axes[1].set_ylabel('Events per week')
axes[1].set_title('Offline Proud Boys activity (US Crisis Monitor)')
axes[1].legend(fontsize=10)

# Mark Jan 6
jan6 = pd.Timestamp('2021-01-06')
for ax in axes:
    ax.axvline(jan6, color='black', linestyle='--', alpha=0.5)
axes[0].text(jan6, axes[0].get_ylim()[1]*0.9, ' Jan 6', fontsize=10)

plt.tight_layout()
plt.show()

print("The January 6 Capitol attack is visible as the largest spike in violent events.")
print("Notice how grievance messaging (red) and offline violence both peak around the same time.")

## 3. Granger Causality: What It Is and What It Isn't

Bailard et al. use **Granger causality** to test whether online grievance messaging predicts future offline violence. Before we replicate this, it's worth understanding what this test actually does.

**Clive Granger** (1934-2009) was a British econometrician who won the Nobel Prize in Economics in 2003. Working in time-series econometrics at UC San Diego, he proposed a practical definition of "causality" for time-series data (Granger, 1969): X "Granger-causes" Y if past values of X help predict future Y, beyond what past Y alone predicts.

**The test is simple**: regress $Y_t$ on its own lags ($Y_{t-1}$), then add lags of $X$ ($X_{t-1}, X_{t-2}$). If adding $X$ lags significantly improves the prediction (F-test), we say X "Granger-causes" Y.

**Why the scare quotes around "causality"?** Granger himself was careful about this. The test checks temporal precedence (X comes before Y) and incremental predictive power (X adds information beyond Y's own past). But temporal precedence is a very weak condition for causation:

- A **shared cause** (confound) can make X precede Y without X causing Y. If a news event triggers both online anger on Monday and offline mobilization on Wednesday, the anger "Granger-causes" the mobilization, but both were caused by the news event.
- The test has **no model of the data-generating process**. In the language of causal graphs (DAGs/PGMs), Granger causality doesn't distinguish between $X \to Y$, $X \leftarrow Z \to Y$, or $X \leftarrow Y$ with different time lags.
- It is, as one critic put it, "just a delay." Adding a time lag to a correlation doesn't make it causal.

**Why use it at all?** When you don't have an instrument, an experiment, or a discontinuity, Granger causality is one of the few tools available. It's honest about its limitations: the name says "causality" but the method delivers prediction. For early warning systems (can we predict violence from online chatter?), prediction is enough. For policy (would shutting down the channels reduce violence?), it isn't.

In [ ]:
import statsmodels.formula.api as smf

# Create lagged variables
df['grievance_lag1'] = df['diagnostic'].shift(1)
df['grievance_lag2'] = df['diagnostic'].shift(2)
df['violence_lag1'] = df['violent_event'].shift(1)

sample = df.dropna(subset=['grievance_lag1', 'violence_lag1']).copy()

# Model 1: violence predicted by past violence only (autoregressive)
m1 = smf.ols('violent_event ~ violence_lag1', data=sample).fit()

# Model 2: add lagged grievance messages
m2 = smf.ols('violent_event ~ violence_lag1 + grievance_lag1', data=sample).fit()

# Model 3: add second lag
m3 = smf.ols('violent_event ~ violence_lag1 + grievance_lag1 + grievance_lag2', data=sample).fit()

print("Does last week's grievance messaging predict this week's violence?")
print()
print(f"{'Model':<45} {'Coef':>8} {'SE':>8} {'p':>8}")
print("-" * 72)
print(f"{'Violence_lag1 only (autoregressive)':<45} {m1.params['violence_lag1']:>8.3f} {m1.bse['violence_lag1']:>8.3f} {m1.pvalues['violence_lag1']:>8.3f}")
print(f"{'+ Grievance_lag1':<45} {m2.params['grievance_lag1']:>8.3f} {m2.bse['grievance_lag1']:>8.3f} {m2.pvalues['grievance_lag1']:>8.3f}")
print(f"{'+ Grievance_lag1 + lag2':<45} {m3.params['grievance_lag1']:>8.3f} {m3.bse['grievance_lag1']:>8.3f} {m3.pvalues['grievance_lag1']:>8.3f}")
print()
print(f"R-squared: autoregressive={m1.rsquared:.3f}, + grievance={m2.rsquared:.3f}, + 2 lags={m3.rsquared:.3f}")
print()
print("If the grievance coefficient is positive and significant, past online")
print("grievance messaging helps predict future offline violence.")

## 4. But Is This Causal?

The regression above shows *prediction*, not necessarily *causation*. Several problems:

**Omitted variables**: maybe a news event (e.g., a court ruling, a counter-protest) causes both more online anger AND more offline mobilization at the same time.

**Reverse causation**: maybe offline events (arrests, clashes) generate online discussion, not the other way around.

**Shared trends**: both online messaging and offline activity could be driven by the same underlying political cycle (elections, trials, legislation).

What would we *need* for a causal claim? Think back to weeks 8-9:
- An **instrument** (something that shifts online messaging without directly affecting offline violence)
- A **natural experiment** (an exogenous shock to one but not the other)
- An **RCT** (randomly assign exposure to messaging... which is ethically impossible here)

In [ ]:
# Visualize the Granger test: does adding grievance lags improve prediction?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: R-squared comparison (the Granger test in a picture)
models = ['Past violence\nonly', '+ 1 lag\ngrievance', '+ 2 lags\ngrievance']
r2s = [m1.rsquared, m2.rsquared, m3.rsquared]
pvals = ['-', f'p={m2.pvalues["grievance_lag1"]:.3f}', f'p={m3.pvalues["grievance_lag1"]:.3f}']
colors = ['gray', '#cc7777', '#cc3333']
bars = axes[0].bar(models, r2s, color=colors, edgecolor='black', alpha=0.8)
for bar, pv in zip(bars, pvals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 pv, ha='center', fontsize=11)
axes[0].set_ylabel('R-squared')
axes[0].set_title('Does online grievance help predict offline violence?')
axes[0].set_ylim(0, max(r2s) * 1.4)

# Right: same test, reverse direction (violence -> grievance)
sample['grievance_lag1_auto'] = sample['diagnostic'].shift(1)
rev1 = smf.ols('diagnostic ~ grievance_lag1_auto', data=sample.dropna(subset=['grievance_lag1_auto'])).fit()
rev2 = smf.ols('diagnostic ~ grievance_lag1_auto + violence_lag1', data=sample.dropna(subset=['grievance_lag1_auto'])).fit()

models_rev = ['Past grievance\nonly', '+ 1 lag\nviolence']
r2s_rev = [rev1.rsquared, rev2.rsquared]
pvals_rev = ['-', f'p={rev2.pvalues["violence_lag1"]:.3f}']
colors_rev = ['gray', '#7777cc']
bars_rev = axes[1].bar(models_rev, r2s_rev, color=colors_rev, edgecolor='black', alpha=0.8)
for bar, pv in zip(bars_rev, pvals_rev):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 pv, ha='center', fontsize=11)
axes[1].set_ylabel('R-squared')
axes[1].set_title('Reverse: does offline violence help predict online grievance?')
axes[1].set_ylim(0, max(r2s_rev) * 1.4)

fig.subplots_adjust(wspace=0.3)
plt.savefig('/tmp/granger-bars.png', dpi=100, bbox_inches='tight')
plt.show()

print("Left: adding grievance lags improves prediction of violence (R2 increases).")
print("Right: adding violence lags does NOT improve prediction of grievance.")
print()
print("This asymmetry is the Granger test result: online grievance helps predict")
print("future violence, but not vice versa. It's suggestive, not causal.")

## 5. Contrast with IV Papers

In weeks 8-9, we saw papers that *could* make causal claims because they had instruments:

| Paper | Instrument | Why it works |
|-------|-----------|-------------|
| Lelkes et al. (2017) | ROW regulations | Laws about cable-laying are unrelated to partisan attitudes |
| Angrist (1990) | Draft lottery | Literally random assignment |

Bailard et al. have no instrument. There is no exogenous shock that affects Proud Boys online messaging without also affecting their offline behavior. This limits what the paper can claim.

**Prediction is still valuable**: knowing that online grievance messaging predicts future violence could help with early warning systems. You don't need causation for prediction. But you do need it to say "if we shut down the Telegram channels, violence would decrease."

## Key Takeaway

The gap between "X predicts Y" and "X causes Y" is the central challenge of causal inference. The methods we've learned (IV, RDD, DiD) are tools for crossing that gap, but they require specific conditions. When those conditions aren't met, we're left with suggestive evidence, not proof.